# Player prop forecaster (Colab GPU tier — Tier 4)

Global probabilistic forecaster (DeepAR) over **all** players at once. For each
player's next game it predicts a *distribution* per stat (mean + sigma), not a
point estimate — which is what the props edge path needs to price over/unders.

Heavy training runs here, never on the laptop. Flow:
1. Export sequences locally: `python run.py --export-sequences --sport nba` (and `--sport nhl`)
2. Upload `data/exports/{sport}_player_sequences.parquet` to Drive `sports-edge/exports/`
3. Run this notebook on a **GPU runtime** (Runtime -> Change runtime type -> GPU)
4. It writes portable, torch-free `{sport}_prop_forecast.pkl` tables back to Drive `sports-edge/artifacts/`
5. Download those into `data/exports/artifacts/` and run `python run.py --import-models`


In [ ]:
# 1. Install deps. GluonTS gives a batteries-included DeepAR; swap for
#    pytorch-forecasting TFT if you prefer. LightGBM/polars are for I/O only.
!pip -q install 'gluonts[torch]' polars pyarrow


In [ ]:
# 2. Mount Drive and locate the exported sequences
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/sports-edge'
EXPORTS = os.path.join(BASE, 'exports')
ARTIFACTS = os.path.join(BASE, 'artifacts')
os.makedirs(ARTIFACTS, exist_ok=True)


In [ ]:
# 3. Confirm GPU is visible
import torch
print('CUDA available:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


In [ ]:
# 4. Which stats each sport's forecaster covers (must match models.train.PROP_STATS)
PROP_STATS = {
    'nba': ['pts', 'reb', 'ast', 'stl', 'blk'],
    'nhl': ['goals', 'assists', 'points', 'shots'],
}
PRED_SAMPLES = 200   # sample paths drawn to estimate mean + sigma


In [ ]:
# 5. Train one global DeepAR per stat, forecast each player's next game,
#    and distill to a portable [player_id, stat, pred_mean, pred_sigma] table.
import polars as pl, numpy as np, pickle
from gluonts.dataset.common import ListDataset
from gluonts.torch import DeepAREstimator

FREQ = 'D'  # daily index; irregular gaps are fine, we model the ordered series

def forecast_sport(sport):
    path = os.path.join(EXPORTS, f'{sport}_player_sequences.parquet')
    assert os.path.exists(path), f'Upload {sport}_player_sequences.parquet to {EXPORTS}'
    df = pl.read_parquet(path).sort(['player_id', 'game_date'])
    asof = str(df['game_date'].max())
    rows = []
    for stat in PROP_STATS[sport]:
        if stat not in df.columns:
            print(f'  [{sport}/{stat}] column missing, skipping'); continue
        # build one series per player (>= 10 games for a usable history)
        series, pids = [], []
        for pid, g in df.group_by('player_id', maintain_order=True):
            vals = g[stat].fill_null(0).to_numpy().astype('float32')
            if len(vals) < 10: continue
            start = str(g['game_date'][0])
            series.append({'start': start, 'target': vals})
            pids.append(pid[0] if isinstance(pid, tuple) else pid)
        if not series:
            print(f'  [{sport}/{stat}] no players with enough history'); continue
        train_ds = ListDataset(series, freq=FREQ)
        est = DeepAREstimator(freq=FREQ, prediction_length=1, num_layers=2,
                              hidden_size=40, trainer_kwargs={'max_epochs': 15})
        predictor = est.train(train_ds)
        forecasts = list(predictor.predict(train_ds, num_samples=PRED_SAMPLES))
        for pid, fc in zip(pids, forecasts):
            s = fc.samples[:, 0]  # next-game sample paths
            rows.append({'player_id': pid, 'stat': stat,
                         'pred_mean': float(np.mean(s)), 'pred_sigma': float(np.std(s))})
        print(f'  [{sport}/{stat}] forecast {len(pids)} players')
    table = pl.DataFrame(rows)
    art = {'sport': sport, 'asof': asof, 'table': table}
    out = os.path.join(ARTIFACTS, f'{sport}_prop_forecast.pkl')
    with open(out, 'wb') as f: pickle.dump(art, f)
    print(f'  wrote {out}  ({table.height} player-stat rows, asof {asof})')
    return art

for sport in ['nba', 'nhl']:
    forecast_sport(sport)


### Serving

The `.pkl` tables are torch-free. Download them into `data/exports/artifacts/`,
run `python run.py --import-models`, and `models.prop_forecast.prop_forecast_features`
will join `fc_{stat}_mean` / `fc_{stat}_sigma` onto the player feature rows. The
distribution (mean + sigma) prices over/unders directly; no torch needed locally.
